# APEX v2: Automated PK/PBPK Parameter Extraction Pipeline

This notebook allows you to run and monitor the **APEX v2** extraction pipeline interactively.

### Pipeline Highlights:
- **Multi-tier Backend:** Automatically routes to Local SGLang, DGX Server (`100.85.128.70:30001`), or OpenAI-compatible API.
- **Hybrid Ingestion:** Uses open-access JATS XML when available (PMC / Europe PMC) with PyMuPDF & Table Transformer fallback.
- **Quality Assurance:** Generates complete parameter records, Tier 1 rule-based verification, and OECD GD 331 regulatory assessment.

### Step 1: Verify Environment & Active Backend

In [ ]:
import os
from pk_pipeline.config import Config
from pk_pipeline.ml_services import SGLangExtractor

print("1. Configuration from .env:")
print(f"   • Local SGLang URL: {Config.SGLANG_LOCAL_URL}")
print(f"   • DGX SGLang URL:   {Config.DGX_SGLANG_URL}")
print(f"   • Output Directory: {Config.OUTPUT_DIR}")

print("\n2. Initializing backend probe...")
extractor = SGLangExtractor()
print(f"   • Backend Selected: {extractor.backend_mode}")
print(f"   • Endpoint URL:     {extractor.server_url}")
print(f"   • Model:            {extractor.model_name}")

### Step 2: Choose Target PDF File

Select any scientific paper PDF to extract. Several test papers are pre-installed in `pk_pipeline/test_data/`.

In [ ]:
from pathlib import Path

# Pre-installed test datasets:
test_data_dir = Path("pk_pipeline/test_data")
available_pdfs = list(test_data_dir.glob("*.pdf"))

print(f"Found {len(available_pdfs)} test papers:")
for i, p in enumerate(available_pdfs, 1):
    print(f"  [{i}] {p.name}")

# Set your target PDF here:
TARGET_PDF = str(available_pdfs[3]) if len(available_pdfs) > 3 else str(available_pdfs[0])
TARGET_COMPOUNDS = ["PFOS", "PFOA"]  # Optional compound steering (e.g. ['Riluzole'] or None)

print(f"\nTarget Paper: {TARGET_PDF}")
print(f"Target Compounds: {TARGET_COMPOUNDS}")

### Step 3: Run the Extraction Pipeline

In [ ]:
import asyncio
from pk_pipeline.main import run_pipeline

print(f"Starting APEX v2 extraction on: {TARGET_PDF}")

# Run extraction (Jupyter notebook handles async natively via top-level await)
doc_result = await run_pipeline(
    pdf_path=TARGET_PDF,
    target_compounds=TARGET_COMPOUNDS,
    use_wevisdoc=True,
    force_pdf=False,   # Set True to bypass PMC XML lookup
    verify=True,
    output_dir="output"
)

print("\n Pipeline Extraction Complete!")

### Step 4: Inspect Extracted Parameters as a DataFrame

In [ ]:
import pandas as pd

params = doc_result.get("model_parameters", [])
print(f"Total Model Parameters Extracted: {len(params)}")

rows = []
for p in params:
    ctx = p.get("context", {})
    qd = p.get("quantitative_data", {})
    prov = p.get("provenance", {})
    rows.append({
        "Symbol": p.get("symbol") or p.get("symbol_standardized"),
        "Name": p.get("parameter_name_standardized") or p.get("parameter_name"),
        "Class": p.get("parameter_class"),
        "Value": qd.get("value"),
        "Unit": qd.get("unit") or qd.get("unit_original"),
        "Species": ctx.get("species") or ctx.get("subject_species"),
        "Compound": ctx.get("compound"),
        "Life Stage": ctx.get("life_stage"),
        "Source": prov.get("source_location") or prov.get("source_table"),
        "Status": prov.get("status")
    })

df_params = pd.DataFrame(rows)
display(df_params.head(25))

# Parameter class distribution:
print("\nParameters by Class:")
print(df_params["Class"].value_counts())

### Step 5: Inspect Verification & OECD Assessment Summary

In [ ]:
classification = doc_result.get("model_classification", {})
print(f"Model Classification: {classification.get('model_type')}")

# Check OECD GD 331 Assessment:
oecd = doc_result.get("oecd_assessment")
if oecd:
    print(f"\nOECD Checklist Version: {oecd.get('checklist_version')}")
    items = oecd.get("items", [])
    print(f"Total Evaluation Items: {len(items)}")
    for item in items[:5]:
        print(f"  • [{item.get('item_id')}] {item.get('question')[:60]}... -> {item.get('response')}")

# Check Tier 1 Verification:
verif = doc_result.get("verification")
if verif:
    print(f"\nVerification Summary:")
    print(f"  • Flagged Issues: {len(verif.get('flagged', []))}")
    for issue in verif.get('flagged', [])[:5]:
        print(f"    - {issue}")